## 5. Grants_Medallion

Asigna permisos de lectura de Unity Catalog sobre el catalogo de futbol a los usuarios del widget `usuarios`
(correos separados por coma): `USE CATALOG`, `USE SCHEMA` en bronze, silver y gold, y `SELECT` sobre sus tablas.

In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text catalogo default "football_dev";

In [0]:
dbutils.widgets.text("usuarios", "cvalenciam@uni.pe")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
usuarios = [u.strip() for u in dbutils.widgets.get("usuarios").split(",") if u.strip()]
if not usuarios:
    raise ValueError("Indica al menos un usuario en el widget 'usuarios'")

TABLAS = {
    "bronze": ["leagues", "matches", "clubs"],
    "silver": ["matches_transformed"],
    "gold": ["season_stats", "standings", "calendar_stats", "club_elo", "match_predictions"],
}

for usuario in usuarios:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {catalogo} TO `{usuario}`")
    for esquema, tablas in TABLAS.items():
        spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalogo}.{esquema} TO `{usuario}`")
        for tabla in tablas:
            spark.sql(f"GRANT SELECT ON TABLE {catalogo}.{esquema}.{tabla} TO `{usuario}`")
    print(f"Permisos otorgados a {usuario}")

In [0]:
display(spark.sql(f"SHOW GRANTS ON CATALOG {catalogo}"))